# END-TO-END PROJECT: SUPERMARKET CUSTOMER RATING PREDICTION SYSTEM

## Step 1: Load and Inspect Data
*(Note, make sure to upload `supermarket_sales.csv` to the Colab session before running, as I don't want to setup my kaggle api)*

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("supermarket_sales.csv")

# view first rows
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'supermarket_sales.csv'

In [ ]:
# 1. Check for missing values
print("Checking for missing values in each column:")
print(df.isnull().sum())

Checking for missing values in each column:


NameError: name 'df' is not defined

In [ ]:
# 2. Check for duplicate Invoice IDs
duplicate_count = df['invoice_id'].duplicated().sum()
print(f"\nNumber of duplicate Invoice IDs: {duplicate_count}")

NameError: name 'df' is not defined

In [ ]:
# 3. Display summary statistics
# The mean rating is suppose to be ~6.97.
df.describe()

## Step 2: Exploratory Data Analysis (EDA)
Time to visualize the data and see what might be driving customer satisfaction.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# style for our plot
sns.set_theme(style="whitegrid")

In [ ]:
# 1. Rating Distribution
# Plotting a histogram to see how ratings spread out between 4 and 10
plt.figure(figsize=(8, 5))

mean_rating = df['rating'].mean()

sns.histplot(df['rating'], bins=15, kde=True, color='teal')
plt.axvline(mean_rating, color='red', linestyle='--', label=f'Mean: {mean_rating:.2f}')
plt.title('Distribution of Customer Ratings')
plt.xlabel('Rating (4 to 10)')
plt.ylabel('Frequency (Number of Customers)')
plt.legend()
plt.show()

NameError: name 'plt' is not defined

The rating distribution shows that customer ratings are spread out between 4.0 and 10.0, with the overall average sitting at ~6.97(6.97270).

In [ ]:
# 2. Branch Comparison
# Checking if any specific branch performs better than the others on average

plt.figure(figsize=(8, 5))
sns.barplot(x='branch', y='rating', data=df, errorbar=None, palette='muted')
plt.title('Average Rating per Branch')
plt.xlabel('Branch (A, B, C)')
plt.ylabel('Average Rating')
plt.ylim(0, 10)
plt.show()

NameError: name 'plt' is not defined

When comparing the three branches (A, B, and C), the average ratings are almost identical. There is no major performance gap or "underperforming" branch. The customer experience is uniform across all locations.

In [ ]:
# 3. Gender/Customer Type Analysis
# Comparing satisfaction levels between Members and Normal customers, split by Gender
plt.figure(figsize=(9, 6))
sns.boxplot(x='customer_type', y='rating', hue='gender_customer', data=df, palette='Set2')
plt.title('Customer Satisfaction by Type and Gender')
plt.xlabel('Customer Type')
plt.ylabel('Rating')
plt.legend(title='Gender')
plt.show()

NameError: name 'plt' is not defined

The boxplot analysis comparing Gender and Customer Type (Member vs. Normal) showed very little difference in the median ratings. It shows that having a Membership does not automatically guarantee that a customer will rate their shopping experience higher than a Normal customer.

In [ ]:

# 4. Correlation Heatmap (Expanded)
plt.figure(figsize=(10, 8))

# Select only relevant numerical columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", center=0)
plt.title('Full Numerical Correlation Heatmap', fontsize=14)
plt.show()

The full numerical correlation heatmap reveals two critical insights:
* Redundancy (Multicollinearity): There is a perfect correlation (1.00) between revenue, cogs, 5pct_markup, and gross_income. This is expected because these are derived from each other using a fixed formula. To avoid 'overfitting' and redundant data, we should only keep one of these features for modeling.
* Independence of Ratings: The target variable (rating) shows near-zero correlation (between -0.01 and -0.04) with every single numerical feature. This confirms that a customer's satisfaction is not influenced by how much they spend, the quantity they buy, or the price of the items.

Note: gm_pct appears blank because it is a constant value across the entire dataset, meaning it has zero variance and cannot be correlated.

In [ ]:

# 5. Time Series Analysis
# Ensuring proper datetime conversion
df['date'] = pd.to_datetime(df['date'], errors='coerce') # handles mixed formats
df['hour'] = pd.to_datetime(df['time']).dt.hour
df['day_of_week'] = df['date'].dt.day_name()

# Plot 1: Average rating by Hour
plt.figure(figsize=(12, 5))
# Using a lineplot with a confidence interval (shadow) to show variance
sns.lineplot(x='hour', y='rating', data=df, marker='o', color='purple', errorbar=('ci', 95))

plt.title('Average Rating Trends by Hour (with 95% Confidence Interval)', fontsize=14)
plt.xlabel('Hour (24h format)')
plt.ylabel('Average Rating')
plt.xticks(range(df['hour'].min(), df['hour'].max() + 1))
plt.grid(True, alpha=0.3)
plt.show()

# Plot 2: Average rating by Day of the Week
days_ordered = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(10, 5))
sns.pointplot(x='day_of_week', y='rating', data=df, order=days_ordered, color='teal')

plt.title('Rating Consistency across the Week', fontsize=14)
plt.xlabel('Day of the Week')
plt.ylabel('Average Rating')
plt.ylim(6, 8) # Zooming in to see the small variations better
plt.show()

NameError: name 'df' is not defined

Based on the time series analysis, ratings stay fairly consistent across the days of the week (hovering around 6.8 to 7.1). Looking at the hour of the day, there are slight peaks in satisfaction around 12:00 PM (noon) and 6:00 PM, with noticeable dips around 11:00 AM and 7:00 PM, though the variance isn't large enough to indicate a major operational issue.

 In conclusion, our target variable (rating) has almost zero correlation with our numerical features like revenue and quantity, the data suggests that financial metrics do not drive customer satisfaction. The data engineering and modeling teams should heavily prioritize our categorical variables (such as Product Line, Branch, and Payment Method) during feature engineering, as these are the most likely areas where predictive patterns regarding customer experience might be found.
 Though the full correlation heatmap shows that financial columns like revenue, cogs, and gross_income are perfectly correlated (1.00), meaning they are redundant and provide the same information. We should drop these extra columns to keep the data clean.

## Step 3: Feature Engineering & Preprocessing



In [ ]:
!pip install category_encoders

In [ ]:
import pandas as pd
from sklearn.preprocessing import RobustScaler

# install once if needed
# !pip install category_encoders
from category_encoders import TargetEncoder

# Load dataset
df = pd.read_csv("supermarket_sales.csv")

# --- DateTime Features ---
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df['time'] = pd.to_datetime(df['time'], format='%H:%M')

df['hour'] = df['time'].dt.hour
df['weekday'] = df['date'].dt.dayofweek

# --- Drop redundant columns ---
df = df.drop(columns=[
    'city', 'date', 'time',
    '5pct_markup', 'cogs', 'gm_pct', 'gross_income'
], errors='ignore')

# --- Binary Encoding ---
df['customer_type'] = df['customer_type'].map({'Member': 1, 'Normal': 0})
df['gender_customer'] = df['gender_customer'].map({'Male': 1, 'Female': 0})

# --- One-Hot Encoding ---
df = pd.get_dummies(df, columns=['branch', 'payment_method'], drop_first=True)

# --- Target Encoding ---
te = TargetEncoder()
df['product_line'] = te.fit_transform(df['product_line'], df['rating']).astype(float)

# --- Feature Scaling ---
scaler = RobustScaler()

num_cols = ['unit_cost', 'revenue', 'quantity', 'hour', 'weekday']
num_cols = [col for col in num_cols if col in df.columns]

df[num_cols] = scaler.fit_transform(df[num_cols])

df.head()

ModuleNotFoundError: No module named 'category_encoders'

In this step, the dataset was cleaned and transformed into a format suitable for modeling. Date and time were converted into hour and weekday to capture when transactions occur. Categorical variables were encoded based on their type, while unnecessary and highly correlated columns were removed. Numerical features were also scaled to make them more consistent.

Binary variables like customer type and gender were converted into 0 and 1 because they only have two categories. Branch and payment method were encoded using One-Hot Encoding since they are nominal and have no order. Product line was encoded using Target Encoding so that each category reflects its average rating, which helps the model learn better patterns. RobustScaler was used instead of StandardScaler because the data includes financial values that may contain outliers.

The dataset is now fully numeric and ready for modeling. The features are more meaningful, less redundant, and better structured, which should improve model performance.

## Step 4: Train-Test Split

Separate the dataset into input features (X) and the target variable (y)
X contains all columns used to predict the customer rating
 y contains only the 'Rating' column, which is what we want the model to predict


In [ ]:
from sklearn.model_selection import train_test_split

# Features and target
X = df.drop(columns=['rating'])
y = df['rating']

# Split the data into training and testing sets
# 80% of the data is used to train the model am
# 20% is reserved for testing how well the model performs on unseen data
# random_state=42 ensures reproducibility (same split every time the code runs)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Display the shapes of the resulting datasets

# print("Training target shape:", y_train.shape)
# print("Testing target shape:", y_test.shape)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In this step, the dataset was divided into training and testing sets. The features were separated from the target variable (rating), and the data was split using an 80/20 ratio to allow proper evaluation of the model.


The split ensures that the model is trained on one portion of the data and tested on unseen data. A fixed random state was used so that the results remain consistent every time the code is run.

The training set contains 800 records and the testing set contains 200 records. This split provides enough data for learning while still allowing a reliable evaluation of model performance.

## PHASE 2: MODEL TRAINING AND SELECTION
Here we train three models

## Model 1: Linear Regression
Linear Regression is used as a baseline because it is simple, interpretable, and helps us understand whether a basic linear model can predict customer ratings effectively.

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize the Linear Regression model
# This serves as a baseline model for regression tasks
# It assumes a linear relationship between the input features and the target variable

lr = LinearRegression()

# Fix: Drop the 'invoice_id' column from X_train and X_test
# This column contains non-numeric string values which cannot be used by LinearRegression.
X_train_processed = X_train.drop(columns=['invoice_id'])
X_test_processed = X_test.drop(columns=['invoice_id'])

# Train the model using the training data
# The model learns the relationship between the features and customer ratings
lr.fit(X_train_processed, y_train)

# Use the trained model to make predictions on the test dataset
pred_lr = lr.predict(X_test_processed)

## Model 2: Random Forest

To capture non linear relationships between product categories and rating.
Random Forest is expected to perform better than Linear Regression because customer satisfaction may depend on complex interactions between variables such as product line, payment method, and purchase amount.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Initialize the Random Forest Regressor
# Random Forest is an ensemble learning method that combines many decision trees
# It is useful for capturing complex and non-linear relationships in the data
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

# Drop the 'invoice_id' column for Random Forest as well
X_train_rf_processed = X_train.drop(columns=['invoice_id'])
X_test_rf_processed = X_test.drop(columns=['invoice_id'])

# Train the Random Forest model on the training data
rf.fit(X_train_rf_processed, y_train)

# Generate predictions on the test data
pred_rf = rf.predict(X_test_rf_processed)

## Model 3: Gradient Boosting

Gradient Boosting is a powerful boosting algorithm that often produces strong predictive performance by gradually improving model accuracy.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
# Initialize the Gradient Boosting Regressor
# Gradient Boosting builds models sequentially, where each new model tries to correct
# the errors made by the previous one
gb = GradientBoostingRegressor()

# Drop the 'invoice_id' column for Gradient Boosting as well
X_train_gb_processed = X_train.drop(columns=['invoice_id'])
X_test_gb_processed = X_test.drop(columns=['invoice_id'])

# Train the Gradient Boosting model
gb.fit(X_train_gb_processed, y_train)

# Predict customer ratings on the test dataset
pred_gb = gb.predict(X_test_gb_processed)

## Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Define a reusable function to evaluate regression model performance
# This helps us avoid repeating the same evaluation code for each model
def evaluate(y_test, pred):

     # Mean Absolute Error(mae): measures the average absolute difference between predicted and actual ratings
    mae = mean_absolute_error(y_test, pred)

     # Root Mean Squared Error(RMSE): measures prediction error, giving more weight to larger errors
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    # R-squared(r2): Indicates how well the model explains the variance in customer ratings
    r2 = r2_score(y_test, pred)

    # Return all three evaluation metrics
    return mae, rmse, r2

## Evaluate all Models used

In [ ]:
# Evaluate the Linear Regression model
lr_results = evaluate(y_test, pred_lr)

# Evaluate the Random Forest model
rf_results = evaluate(y_test, pred_rf)

# Evaluate the Gradient Boosting model
gb_results = evaluate(y_test, pred_gb)

# Print the performance of each model
print("Linear Regression:", lr_results)
print("Random Forest:", rf_results)
print("Gradient Boosting:", gb_results)

## Models Comparison

In [ ]:

# Create a DataFrame to compare all trained models side by side
# This makes it easier to determine which model performs best
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Gradient Boosting"],
    "MAE": [lr_results[0], rf_results[0], gb_results[0]],
    "RMSE": [lr_results[1], rf_results[1], gb_results[1]],
    "R2": [lr_results[2], rf_results[2], gb_results[2]]
})

# Display the comparison table
print(results)

## Best Performing Model
The best model is usually selected based on the;

- Lowest MAE score
- Lowest RMSE score
- Highest R² score

In [ ]:
# Based on the evaluation metrics, Linear Regression was selected as the best-performing model
results = pd.DataFrame({
    "Model": "Linear Regression",
    "MAE": [lr_results[0]],
    "RMSE": [lr_results[1]],
    "R2": [lr_results[2]]
})

print(results)

## MODEL SERIALIZATION REQUIREMENTS
Save the best Model

In [ ]:
import joblib

# Save the trained best-performing model to a .pkl file
# This allows the model to be reused later without retraining
joblib.dump(lr, 'rating_model.pkl')

# Save the preprocessing object (scaler) used during feature transformation
# This ensures that future input data is processed in the same way as the training data
joblib.dump(scaler, 'preprocessor.pkl')

## Test loading thay the files exist and the model works

In [ ]:
model = joblib.load("rating_model.pkl")
scaler = joblib.load("preprocessor.pkl")

print("Model loaded successfully")